In [1]:
partition = 300

In [2]:
import sys
from train import main
from itertools import product  
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


In [3]:
import re

def load_tested_configs(log_path):
    tested = set()
    with open(log_path, 'r') as f:
        for line in f:
            if line.startswith("Running:"):
                match = re.findall(r"[-\w.]+=\S+", line)
                if match:
                    # Normalize values to correct types
                    config = tuple([
                        int(re.search(r"=(\d+)", match[0]).group(1)),       # n_tree
                        int(re.search(r"=(\d+)", match[1]).group(1)),       # t_depth
                        int(re.search(r"=(\d+)", match[2]).group(1)),       # hd
                        int(re.search(r"=(\d+)", match[3]).group(1)),       # batch_size
                        float(re.search(r"=(\d+\.?\d*)", match[4]).group(1)), # feature_rate
                        float(re.search(r"=(\d+\.?\d*)", match[5]).group(1)), # dropout
                        float(re.search(r"=(\d+\.?\d*)", match[6]).group(1)), # lr
                    ])
                    tested.add(config)
    return tested


In [4]:
import random
from itertools import product
import sys
#{'n_tree': 20, 'tree_depth': 10, 'batch_size': 256, 'tree_feature_rate': 0.1, 'feat_dropout': 0.2, 'lr': 0.01}

log_path = f"logs{partition}.txt"
#tested_configs = load_tested_configs(log_path)

n_tree_values = [5, 10, 20, 50, 100]
tree_depth_values = [9, 10, 11, 12]
hidden_dim = [1024, 768]
batch_size_values = [256, 512]
tree_feature_rates = [0.1, 0.2, 0.3, 0.4]
feat_dropouts = [0.0, 0.1, 0.2, 0.3]
lrs = [0.01]

n_iter = 50
best_score = 0
best_config = {}

param_space = list(product(
    n_tree_values,
    tree_depth_values,
    hidden_dim,
    batch_size_values,
    tree_feature_rates,
    feat_dropouts,
    lrs
))

best_acc = 0


#available_configs = [cfg for cfg in param_space if cfg not in tested_configs]
available_configs = [cfg for cfg in param_space]
sampled_configs = random.sample(available_configs, min(n_iter, len(available_configs)))
i = 1
for n_tree, t_depth, hd, batch_size, feature_rate, dropout, lr in sampled_configs:

    log_line = f"Running: n_tree={n_tree}, t_depth={t_depth}, hd={hd}, batch_size={batch_size}, feature_rate={feature_rate}, dropout={dropout}, lr={lr}"
    print(f"\n{log_line}")
    with open(log_path, "a") as log_file:
        log_file.write(f"\n{log_line}\n")

    sys.argv = [
        'train.py',
        '-dataset', f'gtd{partition}',
        '-n_class', '30',
        '-gpuid', '0',
        '-n_tree', str(n_tree),
        '-tree_depth', str(t_depth),
        '-batch_size', str(batch_size),
        '-hidden_dim', str(hd),
        '-tree_feature_rate', str(feature_rate),
        '-feat_dropout', str(dropout),
        '-lr', str(lr),
        '-epochs', '400',
        '-verbose', '0',
        '-jointly_training',
        '-searching', '1'
    ]

    print(f"{i} / 100")
    acc = main()
#print(acc)
    with open(log_path, "a") as log_file:
        log_file.write(f"\n{acc}\n")
    i =i + 1

    if acc > best_acc:
        best_acc = acc
        best_config = {
            'n_tree': n_tree,
            'tree_depth': t_depth,
            'batch_size': batch_size,
            'hidden_dim': hd,
            'tree_feature_rate': feature_rate,
            'feat_dropout': dropout,
            'lr': lr
        }

print("\nBest hyperparameter configuration:")
print(best_config)
print(f"Best accuracy: {best_acc}")



Running: n_tree=100, t_depth=9, hd=768, batch_size=512, feature_rate=0.4, dropout=0.2, lr=0.01
1 / 100
Use gtd300 dataset
Patience: 100


Training Epochs: 100%|██████████| 400/400 [09:56<00:00,  1.49s/it]



Best Accuracy: 0.545238

Running: n_tree=100, t_depth=11, hd=1024, batch_size=256, feature_rate=0.2, dropout=0.0, lr=0.01
2 / 100
Use gtd300 dataset
Patience: 100


Training Epochs:  82%|████████▏ | 328/400 [18:38<04:05,  3.41s/it]

Early stopping at epoch 329

Best Accuracy: 0.565079

Running: n_tree=100, t_depth=11, hd=768, batch_size=256, feature_rate=0.4, dropout=0.3, lr=0.01
3 / 100
Use gtd300 dataset


...


Best Accuracy: 0.551587

Running: n_tree=20, t_depth=12, hd=768, batch_size=256, feature_rate=0.3, dropout=0.3, lr=0.01
49 / 100
Use gtd300 dataset
Patience: 100


Training Epochs: 100%|██████████| 400/400 [04:59<00:00,  1.33it/s]



Best Accuracy: 0.566667

Running: n_tree=50, t_depth=10, hd=1024, batch_size=256, feature_rate=0.2, dropout=0.1, lr=0.01
50 / 100
Use gtd300 dataset
Patience: 100


Training Epochs:  72%|███████▎  | 290/400 [07:26<02:49,  1.54s/it]

Early stopping at epoch 291

Best Accuracy: 0.551587

Best hyperparameter configuration:
{'n_tree': 100, 'tree_depth': 10, 'batch_size': 256, 'hidden_dim': 768, 'tree_feature_rate': 0.1, 'feat_dropout': 0.0, 'lr': 0.01}
Best accuracy: 0.5801587301587302


In [5]:
"""

========== Final Test Evaluation ==========
Model Parameters:
  Dataset: gtd300
  Hidden Dim: 1024
  n_tree: 20, tree_depth: 10, tree_feature_rate: 0.1
  Batch size: 256, Dropout: 0.1, LR: 0.01

Best Accuracy: 0.9070
Weighted Precision: 0.9112, Recall: 0.9070, F1 Score: 0.9062, ROCAUC: 0.9963
Macro Precision: 0.9112, Recall: 0.9070, F1 Score: 0.9062, ROCAUC: 0.9963
Micro Precision: 0.9070, Recall: 0.9070, F1 Score: 0.9070, ROCAUC: 0.9975
"""

'\n\n========== Final Test Evaluation ==========\nModel Parameters:\n  Dataset: gtd300\n  Hidden Dim: 1024\n  n_tree: 20, tree_depth: 10, tree_feature_rate: 0.1\n  Batch size: 256, Dropout: 0.1, LR: 0.01\n\nBest Accuracy: 0.9070\nWeighted Precision: 0.9112, Recall: 0.9070, F1 Score: 0.9062, ROCAUC: 0.9963\nMacro Precision: 0.9112, Recall: 0.9070, F1 Score: 0.9062, ROCAUC: 0.9963\nMicro Precision: 0.9070, Recall: 0.9070, F1 Score: 0.9070, ROCAUC: 0.9975\n'

In [6]:
sys.argv = [
        'train.py',
        '-dataset', f'gtd{partition}',
        '-n_class', '30',
        '-gpuid', '0',
        '-n_tree', str(best_config['n_tree']),
        '-tree_depth', str(best_config['tree_depth']),
        '-batch_size', str(best_config['batch_size']),
        '-hidden_dim', str(best_config['hidden_dim']),
        '-epochs', '1500',
        '-verbose', '0',
        '-tree_feature_rate', str(best_config['tree_feature_rate']),
        '-feat_dropout', str(best_config['feat_dropout']),
        '-lr', str(best_config['lr']),
        '-jointly_training',
        '-searching', '0'
    ]

best_model, preds, targets, labels, epoch_logs = main()

Use gtd300 dataset
Patience: 300


Training Epochs:   3%|▎         | 50/1500 [02:36<1:12:55,  3.02s/it]

[Epoch 50] Train Loss: 1.2573, Eval Loss: 1.7213, Eval Accuracy: 0.5246


Training Epochs:   7%|▋         | 100/1500 [05:12<1:13:12,  3.14s/it]

[Epoch 100] Train Loss: 1.1327, Eval Loss: 1.7169, Eval Accuracy: 0.5500


...

[Epoch 750] Train Loss: 1.0818, Eval Loss: 1.8130, Eval Accuracy: 0.5611


Training Epochs:  53%|█████▎    | 800/1500 [41:06<36:01,  3.09s/it]

[Epoch 800] Train Loss: 1.0852, Eval Loss: 1.8499, Eval Accuracy: 0.5635


Training Epochs:  55%|█████▍    | 819/1500 [42:07<35:01,  3.09s/it]

Early stopping at epoch 820
Evaluating on test set with best model...


In [7]:
from sklearn.metrics import classification_report

print(classification_report(targets, preds))

                                                  precision    recall  f1-score   support

                          Abu Sayyaf Group (ASG)       0.23      0.44      0.31        90
        African National Congress (South Africa)       0.46      0.88      0.61        90
                                Al-Qaida in Iraq       0.40      0.57      0.47        90
        Al-Qaida in the Arabian Peninsula (AQAP)       0.26      0.21      0.23        90
                                      Al-Shabaab       0.19      0.19      0.19        90
             Basque Fatherland and Freedom (ETA)       0.55      0.77      0.64        90
                                      Boko Haram       0.38      0.33      0.36        90
  Communist Party of India - Maoist (CPI-Maoist)       0.54      0.63      0.58        90
       Corsican National Liberation Front (FLNC)       0.67      0.84      0.75        90
                       Donetsk People's Republic       0.50      0.46      0.48        90
Farabundo

In [8]:
def plot_confusion_matrix(y_true, y_pred, labels, partition):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(labels)))
    cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    plt.figure(figsize=(18, 16))
    sns.heatmap(cm_normalized,
                annot=True,
                fmt=".2f",
                xticklabels=labels,
                yticklabels=labels,
                cmap="viridis",
                square=True,
                linewidths=0.5,
                cbar_kws={"shrink": 0.8})

    plt.title(f"Normalized Confusion Matrix (Partition gtd{partition})", fontsize=18)
    plt.xlabel("Predicted Label", fontsize=14)
    plt.ylabel("True Label", fontsize=14)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()

    save_path = f"results/confusion_matrix_partition_gtd{partition}.png"
    plt.savefig(save_path, dpi=300)
    plt.close()

    print(f"Saved confusion matrix for partition gtd{partition} to {save_path}")



In [9]:
plot_confusion_matrix(targets, preds, labels, partition)

ValueError: At least one label specified must be in y_true